################ NOTEBOOK OVERVIEW — V07 ################
# Flow Architecture Overview

V07 adds **video file support (MP4, MKV, MOV, AVI, WEBM)** on top of the V06 base. The Audio
Preprocessor can now accept a URL pointing to a **video file** and extract the audio track
in-memory using an `ffmpeg` subprocess before passing it through the existing DSP chain and
on to the WILLMA Whisper Diarized Transcriber.

All other V06 capabilities are preserved: **built-in Langflow `ChatInput` / `ChatOutput`**,
**pure-Python (stdlib-only) DSP** for WAV files, and **two-speaker diarization** via
`pyannote/speaker-diarization-3.1` with timestamped `[Xm:YYs] Speaker 001/002:` dialogue output.

```
[Chat Input] ──> [1. WILLMA Audio Downloader] ──> [2. Audio Preprocessor (+ffmpeg video)] ──> [3. WILLMA Whisper Diarized Transcriber] ──> [Chat Output (Dialogue Script)]
```

**New in V07:** Component 2 — Audio Preprocessor now detects video containers by file extension
and calls `ffmpeg` (via `subprocess`) to extract the audio track as 16 kHz mono 16-bit PCM WAV
before the DSP chain runs. If `ffmpeg` is unavailable or extraction fails, the original bytes are
forwarded unchanged (resilient passthrough).

> **Docker requirement:** `ffmpeg` must be installed in the Langflow container. Add
> `RUN apt-get update && apt-get install -y ffmpeg && rm -rf /var/lib/apt/lists/*` to your Dockerfile.

**Target environment:** Langflow 1.9.3 · Docker · Ubuntu 22.04 · 8 vCPU · 64 GB RAM

---

## Docker Setup — Enable Video Processing with ffmpeg

V07 requires `ffmpeg` to be available inside the Langflow container. The default
`langflowai/langflow:latest` image does **not** include `ffmpeg`. You must add it to
the `Dockerfile` and rebuild the image before video URLs will work.

---

### Step 1 — Edit the Dockerfile

Open the `Dockerfile` in your `LANGFLOW/` deployment folder on the SRC VM and add the
`ffmpeg` install line. The file should look like this after the edit:

```dockerfile
FROM langflowai/langflow:latest

USER root

# Install system dependencies for Docling
RUN apt-get update && apt-get install -y libgl1 libglib2.0-0

# Install Docling for document parsing (existing line — keep as-is)
RUN pip install --no-cache-dir "docling"

# NEW — Install ffmpeg for video audio extraction (V07)
RUN apt-get update && apt-get install -y --no-install-recommends ffmpeg \
    && rm -rf /var/lib/apt/lists/*

# Switch back to Langflow's default user
USER 1000
```

> **Note:** `--no-install-recommends` keeps the image lean by skipping optional packages.
> `rm -rf /var/lib/apt/lists/*` removes the apt cache afterwards to minimise image size.

---

### Step 2 — Rebuild and restart the Docker stack

SSH into the SRC VM and run the following commands from the `LANGFLOW/` folder:

```bash
# Navigate to the deployment folder
cd ~/LANGFLOW

# Stop the running stack
docker compose down

# Rebuild the Langflow image (picks up the Dockerfile change) and restart all services
docker compose up -d --build
```

The `--build` flag forces Docker to rebuild the `langflow` image from the updated `Dockerfile`.
Traefik and PostgreSQL are unaffected and will restart from their existing images.

---

### Step 3 — Verify ffmpeg is installed inside the container

After the stack is up, confirm ffmpeg is available inside the running Langflow container:

```bash
docker exec langflow ffmpeg -version
```

Expected output (first line):

```
ffmpeg version 4.4.x Copyright (c) 2000-2021 the FFmpeg developers
```

If the command returns `executable file not found`, the image was not rebuilt — re-run
`docker compose up -d --build` and check for build errors with `docker compose logs langflow`.

---

### Step 4 — Verify from inside the Langflow Custom Component

As a quick sanity-check you can also paste this one-liner into a temporary Langflow
**Custom Component** code field and run it via the Playground:

```python
import subprocess
result = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
print(result.stdout.split("\n")[0])  # prints: "ffmpeg version X.X.X ..."
```

If this prints the version string, the Audio Preprocessor's `_extract_audio_from_video()`
method will work correctly for MP4 and other video inputs.

## Input Node: Chat Input (Built-in)

**Purpose:** Acts as the entry point for the user interface.

**Action:** Langflow's built-in `ChatInput` component (internal `lfx.*` API). It captures the text typed into the Langflow Chat Playground — the web URL pointing to your target audio file — and wraps it in a `Message` object before piping it forward. Supports optional file attachments, sender type, session/context IDs, and message history storage. The `input_value` is the field wired to downstream components.

In [ ]:
from lfx.base.data.utils import IMG_FILE_TYPES, TEXT_FILE_TYPES
from lfx.base.io.chat import ChatComponent
from lfx.inputs.inputs import BoolInput
from lfx.io import (
    DropdownInput,
    FileInput,
    MessageTextInput,
    MultilineInput,
    Output,
)
from lfx.schema.message import Message
from lfx.utils.constants import (
    MESSAGE_SENDER_AI,
    MESSAGE_SENDER_NAME_USER,
    MESSAGE_SENDER_USER,
)


class ChatInput(ChatComponent):
    display_name = "Chat Input"
    description = "Get chat inputs from the Playground."
    documentation: str = "https://docs.langflow.org/chat-input-and-output"
    icon = "MessagesSquare"
    name = "ChatInput"
    minimized = True

    inputs = [
        MultilineInput(
            name="input_value",
            display_name="Input Text",
            value="",
            info="Message to be passed as input.",
            input_types=[],
        ),
        BoolInput(
            name="should_store_message",
            display_name="Store Messages",
            info="Store the message in the history.",
            value=True,
            advanced=True,
        ),
        DropdownInput(
            name="sender",
            display_name="Sender Type",
            options=[MESSAGE_SENDER_AI, MESSAGE_SENDER_USER],
            value=MESSAGE_SENDER_USER,
            info="Type of sender.",
            advanced=True,
        ),
        MessageTextInput(
            name="sender_name",
            display_name="Sender Name",
            info="Name of the sender.",
            value=MESSAGE_SENDER_NAME_USER,
            advanced=True,
        ),
        MessageTextInput(
            name="session_id",
            display_name="Session ID",
            info="The session ID of the chat. If empty, the current session ID parameter will be used.",
            advanced=True,
        ),
        MessageTextInput(
            name="context_id",
            display_name="Context ID",
            info="The context ID of the chat. Adds an extra layer to the local memory.",
            value="",
            advanced=True,
        ),
        FileInput(
            name="files",
            display_name="Files",
            file_types=TEXT_FILE_TYPES + IMG_FILE_TYPES,
            info="Files to be sent with the message.",
            advanced=True,
            is_list=True,
            temp_file=True,
        ),
    ]
    outputs = [
        Output(display_name="Chat Message", name="message", method="message_response"),
    ]

    async def message_response(self) -> Message:
        # Ensure files is a list and filter out empty/None values
        files = self.files if self.files else []
        if files and not isinstance(files, list):
            files = [files]
        # Filter out None/empty values
        files = [f for f in files if f is not None and f != ""]

        session_id = self.session_id or self.graph.session_id or ""
        message = await Message.create(
            text=self.input_value,
            sender=self.sender,
            sender_name=self.sender_name,
            session_id=session_id,
            context_id=self.context_id,
            files=files,
        )
        if session_id and isinstance(message, Message) and self.should_store_message:
            stored_message = await self.send_message(
                message,
            )
            self.message.value = stored_message
            message = stored_message

        self.status = message
        return message


## Component 1 — WILLMA Audio Downloader

Downloads the audio file into memory as a `Data` packet (binary `bytes` + filename).

### Supported URL types

| URL pattern | Handled as |
|---|---|
| Direct download (`.mp3`, `.wav`, `.mp4`, …) | Streamed directly |
| GitHub web URL (`/blob/`) | Rewritten to raw.githubusercontent.com |
| Nextcloud / ownCloud file viewer (`/apps/files/files/{id}`) | Rewritten to `/index.php/f/{id}?download` |
| **Public share link** (`/s/{TOKEN}/download?path=…`) | See authentication section below |

### Password-protected share links (SURF Research Drive)

SURF Research Drive is based on **ownCloud**. For password-protected public shares the server **ignores HTTP Basic Auth** on the `/s/{TOKEN}/download` endpoint. The only reliable method is to replicate exactly what a browser does:

1. **GET** `/s/{TOKEN}` — loads the share landing page and extracts the CSRF `requesttoken` from the HTML.
2. **POST** `/s/{TOKEN}/authenticate/downloadshare` — sends `password=…` + `requesttoken=…` → server sets a session cookie.
3. **GET** the original download URL — `requests.Session` sends the cookie automatically, server returns the file.

This flow is implemented in `_authenticated_session_download()`. The `requests.Session` object handles cookie persistence across all three steps.

### Error codes returned via `Data.text`

| Code | Meaning |
|---|---|
| `AUTH_REQUIRED` | Server returned an HTML page (wrong / missing password, or login-gated link) |
| `DOWNLOAD_FAILED` | HTTP error (not 401/403) or network exception |
| `INVALID_URL` | Input is not a valid `http(s)://` URL |

### Inputs
- **Chat Input Message** — URL pasted by the user in the chat
- **Share Link Password (optional)** — password for password-protected SURF Research Drive share links; leave blank for public (no-password) shares
- **EXAMPLE: https://hr.data.surf.nl/s/7Gz8JCzZDfCy42D/download?path=/ECO_LARS_INTERVIEW.mp4   derived from Share link https://hr.data.surf.nl/s/7Gz8JCzZDfCy42D

![SURF Research Drive share link example](images/surf_share_link_example.png)


In [ ]:
import re
import requests
import os
from urllib.parse import urlparse, parse_qs
from langflow.custom import Component
from langflow.inputs import MessageInput, SecretStrInput
from langflow.io import Output
from langflow.schema import Data

class WillmaAudioDownloader(Component):
    display_name = "1. WILLMA Audio Downloader"
    description = "Downloads an audio URL directly from Chat Input into an in-memory Data packet stream."
    
    inputs = [
        MessageInput(
            name="chat_message",
            display_name="Chat Input Message",
            required=True
        ),
        SecretStrInput(
            name="share_password",
            display_name="Share Link Password (optional)",
            value="",
            info="Password for password-protected Nextcloud / SURF Research Drive share links. Leave empty for public (no-password) shares."
        )
    ]
    
    outputs = [
        Output(name="audio_packet", display_name="Audio Packet Data", method="download_to_memory")
    ]

    def _authenticated_session_download(self, url_target, nc_share_token, pwd, parsed_url):
        """Download a file from a password-protected Nextcloud/ownCloud public share.

        Mirrors exactly what a browser does:
          1. GET /s/{token}  →  extract CSRF requesttoken from HTML
          2. POST /s/{token}/authenticate/downloadshare  →  server sets session cookie
          3. GET the original download URL  →  session cookie is sent automatically

        This works for all Nextcloud/ownCloud versions, including SURF Research Drive
        (ownCloud-based).  HTTP Basic Auth on /s/{token}/download is NOT honoured by
        ownCloud/Nextcloud for password-protected public shares — cookie auth is required.
        """
        session = requests.Session()
        base = f"{parsed_url.scheme}://{parsed_url.netloc}"

        # Step 1 — load the share landing page to get the CSRF requesttoken
        request_token = ""
        try:
            page_resp = session.get(
                f"{base}/s/{nc_share_token}", timeout=30, allow_redirects=True
            )
            m = re.search(r'data-requesttoken="([^"]+)"', page_resp.text)
            if m:
                request_token = m.group(1)
        except Exception:
            pass

        # Step 2 — POST the password to the authenticate endpoint.
        # Try the Nextcloud path first, then the ownCloud / generic fallbacks.
        post_data = {"password": pwd}
        if request_token:
            post_data["requesttoken"] = request_token
        auth_headers = {"X-Requested-With": "XMLHttpRequest"}

        for auth_url in [
            f"{base}/s/{nc_share_token}/authenticate/downloadshare",   # Nextcloud
            f"{base}/index.php/s/{nc_share_token}/authenticate",        # ownCloud
            f"{base}/s/{nc_share_token}/authenticate",                  # generic
        ]:
            try:
                session.post(
                    auth_url, data=post_data, headers=auth_headers,
                    timeout=30, allow_redirects=True
                )
                break   # first POST that doesn't throw sets the session cookie
            except Exception:
                continue

        # Step 3 — GET the download URL; the session cookie is sent automatically
        return session.get(url_target, timeout=120, allow_redirects=True)

    def download_to_memory(self) -> Data:
        message_obj = self.chat_message
        url_target = ""
        nc_share_token = None

        if message_obj and hasattr(message_obj, "text") and message_obj.text:
            url_target = str(message_obj.text).strip()
            
        if not url_target or not url_target.startswith(("http://", "https://")):
            return Data(text="INVALID_URL", data={"bytes": None, "filename": "audio.mp3"})

        # --- URL rewriting ---

        # GitHub: rewrite web URLs to raw content
        if "github.com" in url_target and "/blob/" in url_target:
            url_target = url_target.replace("github.com", "raw.githubusercontent.com").replace("/blob/", "/")
        elif "github.com" in url_target and "/raw/" in url_target:
            url_target = url_target.replace("github.com", "raw.githubusercontent.com").replace("/raw/", "/")

        # Nextcloud / SURF Research Drive — file viewer URL
        # /apps/files/files/{id}?dir=... → /index.php/f/{id}?download
        nc_viewer = re.search(r'/apps/files/files/(\d+)', url_target)
        if nc_viewer:
            base = url_target[:url_target.index('/apps/files/files/')]
            url_target = f"{base}/index.php/f/{nc_viewer.group(1)}?download"

        # Nextcloud / ownCloud — public share link.
        # Capture share token; append /download if not already present.
        # ?path= query params are preserved when /download is already in the path.
        else:
            nc_share_match = re.search(r'/s/([A-Za-z0-9_-]+)', url_target.split('?')[0])
            if nc_share_match:
                nc_share_token = nc_share_match.group(1)
                share_base = url_target.split('?')[0].rstrip('/')
                if not share_base.endswith('/download'):
                    url_target = share_base + "/download"

        # Resolve password — handle both plain str and Pydantic SecretStr from Langflow
        pwd = getattr(self, "share_password", "") or ""
        if hasattr(pwd, "get_secret_value"):
            pwd = pwd.get_secret_value()
        pwd = str(pwd).strip() if pwd else ""

        try:
            if nc_share_token and pwd:
                # Cookie-based session auth — the only reliable method for
                # password-protected public shares on Nextcloud / ownCloud / SURF Research Drive.
                parsed = urlparse(url_target)
                response = self._authenticated_session_download(
                    url_target, nc_share_token, pwd, parsed
                )
            else:
                response = requests.get(url_target, timeout=120, allow_redirects=True)

            response.raise_for_status()

            # HTML response → still on the password-entry page (wrong/missing password)
            # or an internal login-required page.
            content_type = response.headers.get("Content-Type", "").lower()
            if "text/html" in content_type:
                return Data(text="AUTH_REQUIRED", data={"bytes": None, "filename": "audio.mp3"})

            audio_bytes = response.content

            # Prefer Content-Disposition filename (reliable for Nextcloud downloads)
            filename = ""
            content_disp = response.headers.get("Content-Disposition", "")
            if content_disp:
                cd_match = re.search(
                    r"filename\*?=(?:UTF-8'')?[\"']?([^\"';\r\n]+)[\"']?",
                    content_disp, re.IGNORECASE
                )
                if cd_match:
                    filename = cd_match.group(1).strip().strip("\"'")

            # Fall back to URL path basename
            if not filename:
                clean_url_path = url_target.split("?")[0]
                filename = os.path.basename(clean_url_path) or "audio.wav"

            return Data(
                text=filename,
                data={
                    "bytes": audio_bytes,
                    "filename": filename,
                    "size_bytes": len(audio_bytes)
                }
            )
        except requests.exceptions.HTTPError as http_err:
            status = http_err.response.status_code if http_err.response is not None else 0
            if status in (401, 403):
                return Data(text="AUTH_REQUIRED", data={"bytes": None, "filename": "audio.mp3"})
            return Data(text="DOWNLOAD_FAILED", data={"bytes": None, "filename": "audio.mp3"})
        except Exception:
            return Data(text="DOWNLOAD_FAILED", data={"bytes": None, "filename": "audio.mp3"})


## 2. Audio Preprocessor (Pure Python / Stdlib + ffmpeg for video)

**Purpose:** Cleans the downloaded audio/video in-memory before transcription. Better audio quality directly improves Whisper accuracy — especially for noisy recordings, telephone audio, or files at non-standard sample rates. Uses **only Python stdlib** (`wave`, `struct`, `math`, `subprocess`, `tempfile`) for all processing — the only external dependency is the system `ffmpeg` binary required for video extraction.

**Key Operations:**

- **Video detection & audio extraction (V07 new):** Files with video extensions (`.mp4`, `.mkv`, `.avi`, `.mov`, `.webm`, `.m4v`, `.mpg`, `.mpeg`) are automatically detected by filename. The `_extract_audio_from_video()` method writes the raw bytes to a `/tmp` temp file, runs `ffmpeg` against it, and reads the resulting WAV back into memory — then deletes both temp files. Using a temp file (rather than stdin pipe) gives ffmpeg full random-access seeking, which is required for MP4 files whose `moov` atom is at the end (common in Zoom recordings, phone captures, and screen recordings). If `ffmpeg` is unavailable or returns a non-zero exit code, the original bytes are forwarded unchanged to the Transcriber.
- **WAV-only DSP chain:** Only the extracted (or directly downloaded) **16-bit PCM `.wav`** files are processed through the DSP chain. All other formats (`mp3`, `m4a`, `flac`) and non-16-bit WAVs are forwarded unchanged.
- **Stereo → mono downmix:** Averages all interleaved channels into a single mono float stream before any DSP.
- **Resampling:** Linear-interpolation upsampling/downsampling to the target sample rate (default **16 kHz**). Whisper is trained on 16 kHz mono audio. For video-extracted audio this is a no-op since `ffmpeg` already outputs 16 kHz.
- **High-pass filter:** First-order RC filter at **90 Hz** removes DC offset, low-frequency rumble, and HVAC noise.
- **Dynamic Range Compression (DRC):** Power-law soft compression (`sign × |s|^0.88`) brings up quiet speech without clipping loud transients.
- **Peak normalization:** Scales all samples so the loudest peak reaches **0.98**, maximising dynamic range without clipping.
- **Re-encoding:** Converts the cleaned float array back to 16-bit PCM WAV bytes using `wave` + `struct`.
- **Resilient fallback:** If any step raises an exception (corrupt header, unexpected encoding, `ffmpeg` not found, etc.), the original `incoming_packet` is returned unchanged so the flow never breaks.

> **Docker requirement:** `ffmpeg` must be present in the Langflow container. Add the following line to your `Dockerfile` (before `USER 1000`):
> ```dockerfile
> RUN apt-get update && apt-get install -y ffmpeg && rm -rf /var/lib/apt/lists/*
> ```

In [ ]:
import io
import os
import tempfile
import wave
import struct
import math
import subprocess
from langflow.custom import Component
from langflow.inputs import DataInput, IntInput, FloatInput, BoolInput
from langflow.io import Output
from langflow.schema import Data

class AudioPreprocessorComponent(Component):
    display_name = "2. Audio Preprocessor"
    description = "Extracts audio from video containers (MP4/MKV/MOV/AVI/WEBM) via ffmpeg, then applies 16kHz resampling, Highpass Filtering, DRC, and Peak Normalization in memory."

    VIDEO_EXTENSIONS = (".mp4", ".mkv", ".avi", ".mov", ".webm", ".m4v", ".mpg", ".mpeg")

    inputs = [
        DataInput(
            name="incoming_packet",
            display_name="Raw Audio/Video Packet Data",
            required=True
        ),
        IntInput(
            name="target_sample_rate",
            display_name="Target Sample Rate (Hz)",
            value=16000
        ),
        FloatInput(
            name="highpass_cutoff_hz",
            display_name="Highpass Cutoff (Hz)",
            value=90.0
        ),
        BoolInput(
            name="enable_drc",
            display_name="Enable Dynamic Range Control",
            value=True
        ),
        FloatInput(
            name="peak_normalize",
            display_name="Peak Normalization Target",
            value=0.98
        )
    ]

    outputs = [
        Output(name="cleaned_packet", display_name="Cleaned Audio Packet", method="preprocess_audio")
    ]

    def _extract_audio_from_video(self, video_bytes: bytes, filename: str = "video.mp4") -> bytes:
        """Extract audio track from a video container via a /tmp temp file.

        Writes video_bytes to a named temp file, runs ffmpeg against it, reads
        the output WAV back into memory, then deletes both temp files.

        Using a real file (instead of stdin pipe) gives ffmpeg full random-access
        seeking — required for MP4 files whose moov atom is at the end (Zoom
        recordings, phone captures, screen recordings).  The cache:pipe approach
        fails for these files inside Docker containers.

        Raises RuntimeError if ffmpeg exits with a non-zero return code.
        """
        ext = filename.rsplit(".", 1)[-1].lower() if "." in filename else "mp4"
        with tempfile.NamedTemporaryFile(suffix=f".{ext}", delete=False, dir="/tmp") as f_in:
            f_in.write(video_bytes)
            tmp_in = f_in.name
        tmp_out = tmp_in + ".wav"
        try:
            cmd = [
                "ffmpeg", "-y",
                "-i", tmp_in,
                "-vn",                                    # drop video stream
                "-ac", "1",                               # force mono
                "-ar", str(self.target_sample_rate),      # resample to target rate
                "-sample_fmt", "s16",                     # 16-bit signed PCM
                "-f", "wav",
                tmp_out
            ]
            result = subprocess.run(cmd, capture_output=True, timeout=300)
            if result.returncode != 0:
                error_msg = result.stderr.decode("utf-8", errors="replace")
                raise RuntimeError(f"ffmpeg audio extraction failed: {error_msg[-500:]}")
            with open(tmp_out, "rb") as f_out:
                return f_out.read()
        finally:
            for p in (tmp_in, tmp_out):
                try:
                    os.unlink(p)
                except OSError:
                    pass

    def _resample_linear(self, data, orig_sr, target_sr):
        if orig_sr == target_sr or len(data) == 0:
            return data
        ratio = float(orig_sr) / float(target_sr)
        target_length = int(len(data) / ratio)
        resampled = [0.0] * target_length
        for i in range(target_length):
            orig_idx = i * ratio
            idx_low = int(math.floor(orig_idx))
            idx_high = min(idx_low + 1, len(data) - 1)
            weight = orig_idx - idx_low
            resampled[i] = (1.0 - weight) * data[idx_low] + weight * data[idx_high]
        return resampled

    def _apply_highpass(self, data, sample_rate, cutoff):
        if len(data) == 0 or cutoff <= 0:
            return data
        rc = 1.0 / (2.0 * math.pi * cutoff)
        dt = 1.0 / sample_rate
        alpha = rc / (rc + dt)
        filtered = [0.0] * len(data)
        filtered[0] = data[0]
        for i in range(1, len(data)):
            filtered[i] = alpha * (filtered[i-1] + data[i] - data[i-1])
        return filtered

    def _apply_drc(self, data):
        boosted = []
        for sample in data:
            sign = 1.0 if sample >= 0 else -1.0
            boosted.append(sign * math.pow(abs(sample), 0.88))
        return boosted

    def preprocess_audio(self) -> Data:
        packet = self.incoming_packet.data if self.incoming_packet else {}
        raw_bytes = packet.get("bytes")
        filename = packet.get("filename", "audio.wav")

        if not raw_bytes or len(raw_bytes) == 0:
            return self.incoming_packet  # passthrough — preserves error codes (AUTH_REQUIRED, DOWNLOAD_FAILED, etc.)

        # Step 0: Extract audio track from video containers (MP4, MKV, MOV, AVI, WEBM, etc.)
        name_lower = filename.lower()
        if any(name_lower.endswith(ext) for ext in self.VIDEO_EXTENSIONS):
            try:
                raw_bytes = self._extract_audio_from_video(raw_bytes, filename)
                # Rename to .wav so the DSP chain picks it up
                filename = filename.rsplit(".", 1)[0] + ".wav"
            except Exception:
                return self.incoming_packet  # ffmpeg unavailable or failed — passthrough to Transcriber

        # Process as WAV if extension matches, otherwise fallback pass-through (e.g. MP3, FLAC)
        if not filename.lower().endswith(".wav"):
            return self.incoming_packet

        try:
            # Decode PCM_16 bytes to Float Mono array
            with wave.open(io.BytesIO(raw_bytes), 'rb') as wav_in:
                orig_sr = wav_in.getframerate()
                channels = wav_in.getnchannels()
                sampwidth = wav_in.getsampwidth()
                n_frames = wav_in.getnframes()
                raw_frames = wav_in.readframes(n_frames)

                if sampwidth != 2:
                    return self.incoming_packet  # Dynamic pass-through if not 16-bit

                fmt = f"<{n_frames * channels}h"
                integer_samples = struct.unpack(fmt, raw_frames)
                float_samples = [s / 32768.0 for s in integer_samples]

            # Downmix Stereo/Surround to Mono channel layout
            if channels > 1:
                mono = []
                for i in range(0, len(float_samples), channels):
                    mono.append(sum(float_samples[i:i+channels]) / channels)
                float_samples = mono

            # Execute DSP chain
            float_samples = self._resample_linear(float_samples, orig_sr, self.target_sample_rate)
            float_samples = self._apply_highpass(float_samples, self.target_sample_rate, self.highpass_cutoff_hz)

            if self.enable_drc:
                float_samples = self._apply_drc(float_samples)

            peak = max(abs(s) for s in float_samples) if float_samples else 0.0
            if peak > 0:
                float_samples = [(s / peak) * self.peak_normalize for s in float_samples]

            # Re-encode clean Float array back into standard PCM_16 WAV bytes
            out_buffer = io.BytesIO()
            with wave.open(out_buffer, 'wb') as wav_out:
                wav_out.setnchannels(1)
                wav_out.setsampwidth(2)
                wav_out.setframerate(self.target_sample_rate)
                int_frames = [int(max(-32768, min(32767, s * 32767))) for s in float_samples]
                wav_out.writeframes(struct.pack(f"<{len(int_frames)}h", *int_frames))

            processed_bytes = out_buffer.getvalue()
            return Data(
                text=filename,
                data={
                    "bytes": processed_bytes,
                    "filename": filename,
                    "size_bytes": len(processed_bytes)
                }
            )
        except Exception:
            return self.incoming_packet  # Resilient fallback in case header tracking breaks


## 3. WILLMA Whisper Diarized Transcriber

**Purpose:** Transcribes the preprocessed audio via the SURF WILLMA Whisper API **and** performs two-speaker diarization, producing a timestamped dialogue script in the Chat Playground.

**Key Operations:**

- **Model Resolution:** Queries `GET /sequences` to find the best available Whisper model (priority: `whisper-large-v3` → `faster-whisper-large-v3` → `whisper-large` → `whisper-medium`).
- **Chunked Processing Window:** The WAV file is split into sequential `chunk_seconds`-wide windows (default **30 s**) using stdlib `wave`. Each chunk is processed independently — STT + diarization — before results are merged.
- **Speech-To-Text (STT):** Each chunk is posted to `/audio/transcriptions` with `verbose_json` + `segment` timestamp granularity, yielding word-level timestamped segments.
- **Speaker Diarization:** Each chunk is simultaneously posted to `/audio/custom-diarization` using the `pyannote/speaker-diarization-3.1` model. If the diarization API fails or returns empty results, a **fallback turn-detector** (`_alternative_diarization_from_stt`) groups STT segments by pause gaps (`pause_threshold`, default **1.2 s**) and alternates speaker labels `001` / `002`.
- **Overlap-Matrix Alignment:** Each STT segment is assigned a speaker by computing the time overlap with every diarization segment; the speaker with the maximum overlap (≥ `min_overlap_seconds`, default **0.15 s**) wins. Segments with no sufficient overlap are labelled `UNKNOWN`.
- **Timeline Stitching:** Segment `start`/`end` times are offset by the chunk's position in the original audio, producing a continuous absolute timeline across all chunks.
- **Post-Processing Pipeline:**
  1. `_force_two_speaker_labels` — collapses all detected speaker IDs to exactly `001` and `002` (by frequency rank).
  2. `_assign_unknown_by_neighbors` — fills `UNKNOWN` segments by borrowing the speaker label from surrounding context.
  3. `_merge_speaker_segments` — merges consecutive same-speaker segments separated by ≤ 1 s into single blocks.
- **Dialogue Script Formatting:** The merged conversation is rendered as a readable script:
  ```
  [Xm:YYs] Speaker 001:
  "..."

  [Xm:YYs] Speaker 002:
  "..."
  ```
- **Result Caching:** `process_audio_pipeline()` caches its result in `_cached_result` so both outputs share one full pipeline run.
- **Two Outputs:**
  - `transcript_message` — formatted dialogue script as a `Message` for the Chat Playground.
  - `segments_json` — list of merged, speaker-labelled segment dicts (`start`, `end`, `text`, `speaker`, `speaker_source`, `speaker_overlap`) for downstream use.

In [ ]:
import io
import wave
import requests
from langflow.custom import Component
from langflow.inputs import DataInput, StrInput, SecretStrInput, IntInput, FloatInput
from langflow.io import Output
from langflow.schema.message import Message

class WillmaWhisperTranscriber(Component):
    display_name = "3. WILLMA Whisper Diarized Transcriber"
    description = "Transcribes and diarizes audio stream segments into timestamped two-speaker dialogue scripts."

    inputs = [
        DataInput(
            name="incoming_packet",
            display_name="Audio Packet Data",
            required=True
        ),
        StrInput(
            name="base_url",
            display_name="WILLMA Base URL",
            value="https://willma.surf.nl/api/v0"
        ),
        SecretStrInput(
            name="api_key",
            display_name="WILLMA API Key",
            required=True
        ),
        StrInput(
            name="language",
            display_name="Language Code",
            value="nl"
        ),
        IntInput(
            name="chunk_seconds",
            display_name="Processing Window Size (s)",
            value=30
        ),
        FloatInput(
            name="min_overlap_seconds",
            display_name="Minimum Speaker Overlap (s)",
            value=0.15
        ),
        FloatInput(
            name="pause_threshold",
            display_name="Diarization Turn Pause Gap (s)",
            value=1.2
        )
    ]

    outputs = [
        Output(name="transcript_message", display_name="Diarized Transcript", method="get_chat_message"),
        Output(name="segments_json", display_name="Structured Segments List", method="get_segments")
    ]

    _MIME_MAP = {
        "mp4": "video/mp4", "mkv": "video/x-matroska", "avi": "video/x-msvideo",
        "mov": "video/quicktime", "webm": "video/webm", "m4v": "video/mp4",
        "mp3": "audio/mpeg", "m4a": "audio/mp4", "flac": "audio/flac",
        "ogg": "audio/ogg", "aac": "audio/aac", "opus": "audio/opus",
        "wav": "audio/wav",
    }

    def _fetch_model_name(self) -> str:
        headers = {"X-API-KEY": self.api_key, "Content-Type": "application/json"}
        resp = requests.get(f"{self.base_url}/sequences", headers=headers, timeout=45)
        resp.raise_for_status()
        catalog = resp.json()
        preferred = ["whisper-large-v3", "faster-whisper-large-v3", "whisper-large", "whisper-medium"]
        for frag in preferred:
            for item in catalog:
                if frag in str(item.get("name", "")).lower():
                    return item.get("name")
        for item in catalog:
            if "whisper" in str(item.get("name", "")).lower():
                return item.get("name")
        return "whisper-large-v3"

    def _overlap_seconds(self, start_a, end_a, start_b, end_b):
        return max(0.0, min(end_a, end_b) - max(start_a, start_b))

    def _build_turns_from_stt(self, segments, pause_threshold):
        turns = []
        if not segments:
            return turns
        sorted_segs = sorted(segments, key=lambda x: (float(x.get("start", 0)), float(x.get("end", 0))))
        current = {
            "start": float(sorted_segs[0].get("start", 0)),
            "end": float(sorted_segs[0].get("end", 0)),
            "text": (sorted_segs[0].get("text") or "").strip(),
        }
        for seg in sorted_segs[1:]:
            seg_start = float(seg.get("start", 0))
            seg_end = float(seg.get("end", 0))
            seg_text = (seg.get("text") or "").strip()
            gap = seg_start - current["end"]
            if gap >= pause_threshold:
                turns.append(current)
                current = {"start": seg_start, "end": seg_end, "text": seg_text}
            else:
                current["end"] = max(current["end"], seg_end)
                if seg_text:
                    current["text"] = (current["text"] + " " + seg_text).strip()
        turns.append(current)
        return turns

    def _alternative_diarization_from_stt(self, segments, pause_threshold, start_with="001"):
        turns = self._build_turns_from_stt(segments, pause_threshold)
        speaker_cycle = ["002", "001"] if start_with == "002" else ["001", "002"]
        diarization = []
        for index, turn in enumerate(turns):
            diarization.append({
                "start": round(float(turn["start"]), 3),
                "end": round(float(turn["end"]), 3),
                "speaker": speaker_cycle[index % 2]
            })
        return diarization

    def _force_two_speaker_labels(self, segments):
        counts = {}
        for seg in segments:
            raw = seg.get("speaker")
            if raw in (None, "", "UNKNOWN"):
                continue
            counts[str(raw).strip()] = counts.get(str(raw).strip(), 0) + 1
        top_speakers = [item[0] for item in sorted(counts.items(), key=lambda x: x[1], reverse=True)[:2]]

        spk_map = {}
        if len(top_speakers) >= 1: spk_map[top_speakers[0]] = "001"
        if len(top_speakers) >= 2: spk_map[top_speakers[1]] = "002"

        forced = []
        for seg in segments:
            updated = dict(seg)
            raw = updated.get("speaker")
            if raw in (None, "", "UNKNOWN"):
                updated["speaker"] = "UNKNOWN"
            else:
                k = str(raw).strip()
                updated["speaker"] = k if k in ("001", "002") else spk_map.get(k, "UNKNOWN")
            forced.append(updated)
        return forced

    def _assign_unknown_by_neighbors(self, segments):
        if not segments:
            return segments
        count = len(segments)
        for index, seg in enumerate(segments):
            if seg.get("speaker") != "UNKNOWN":
                continue
            prev_spk, next_spk = None, None
            for left in range(index - 1, -1, -1):
                if segments[left].get("speaker") and segments[left].get("speaker") != "UNKNOWN":
                    prev_spk = segments[left].get("speaker")
                    break
            for right in range(index + 1, count):
                if segments[right].get("speaker") and segments[right].get("speaker") != "UNKNOWN":
                    next_spk = segments[right].get("speaker")
                    break
            if prev_spk and prev_spk == next_spk:
                seg["speaker"] = prev_spk
        return segments

    def _merge_speaker_segments(self, segments, max_gap=1.0):
        merged = []
        sorted_segs = sorted(segments, key=lambda x: (float(x.get("start", 0)), float(x.get("end", 0))))
        for seg in sorted_segs:
            text = seg.get("text", "").strip()
            if not text:
                continue
            if not merged:
                merged.append(dict(seg))
                continue
            prev = merged[-1]
            same_speaker = prev.get("speaker") == seg.get("speaker")
            gap = float(seg.get("start", 0)) - float(prev.get("end", 0))
            if same_speaker and gap <= max_gap:
                prev["end"] = max(prev["end"], seg.get("end", prev["end"]))
                prev["text"] = (prev.get("text").strip() + " " + text).strip()
            else:
                merged.append(dict(seg))
        return merged

    def _assign_speakers(self, stt_segments, diar_diarization, diar_source, chunk_offset=0.0):
        """Map each STT segment to a speaker via time-overlap matching, then offset timestamps."""
        result = []
        for seg in stt_segments:
            local_start = float(seg.get("start", 0))
            local_end = float(seg.get("end", 0))
            best_speaker = None
            best_overlap = 0.0
            for diar_seg in diar_diarization:
                d_start = float(diar_seg.get("start", 0))
                d_end = float(diar_seg.get("end", 0))
                overlap = self._overlap_seconds(local_start, local_end, d_start, d_end)
                if overlap > best_overlap:
                    best_overlap = overlap
                    best_speaker = diar_seg.get("speaker") or diar_seg.get("label") or diar_seg.get("id")
            out = dict(seg)
            out["start"] = round(local_start + chunk_offset, 3)
            out["end"] = round(local_end + chunk_offset, 3)
            if diar_source == "stt_turn_fallback":
                out["speaker"] = best_speaker or "UNKNOWN"
            else:
                out["speaker"] = best_speaker if best_overlap >= self.min_overlap_seconds else "UNKNOWN"
            out["speaker_source"] = diar_source
            out["speaker_overlap"] = round(best_overlap, 3)
            result.append(out)
        return result

    def process_audio_pipeline(self):
        if hasattr(self, "_cached_result"):
            return self._cached_result

        packet = self.incoming_packet.data if self.incoming_packet else {}
        audio_bytes = packet.get("bytes")
        filename = packet.get("filename", "audio.wav")

        if not audio_bytes or len(audio_bytes) == 0:
            # Surface specific error codes from the downloader as actionable messages
            packet_status = getattr(self.incoming_packet, "text", "") or ""
            if packet_status == "AUTH_REQUIRED":
                return (
                    "Download blocked — authentication required.\n\n"
                    "The URL you pasted is an internal link (requires a SURF login), a folder share, "
                    "or a password-protected share link.\n\n"
                    "How to fix this:\n\n"
                    "  Option A — Share the file directly (no password):\n"
                    "    1. In SURF Research Drive, right-click the AUDIO FILE (not the folder)\n"
                    "    2. Share → New share link → no password → Copy link\n"
                    "    3. Paste that link here (e.g. https://hr.data.surf.nl/s/AbCdEfGh)\n\n"
                    "  Option B — File within a shared folder (no password):\n"
                    "    Append the filename to the folder download URL:\n"
                    "    https://hr.data.surf.nl/s/<sharetoken>/download?path=/filename.mp4\n\n"
                    "  Option C — Password-protected share link:\n"
                    "    Enter the share password in the 'Share Link Password' field\n"
                    "    of the WILLMA Audio Downloader component.",
                    []
                )
            return "No input audio data detected.", []

        stt_model_name = self._fetch_model_name()
        diar_model_name = "pyannote/speaker-diarization-3.1"
        headers = {"X-API-KEY": self.api_key, "Accept": "application/json"}

        all_aligned_segments = []

        # --- Probe: is this a valid WAV container? ---
        # If ffmpeg is unavailable in the container, the preprocessor passes through raw
        # video bytes (MP4/MKV/etc.). Detect this and route to the direct-send path so
        # WILLMA Whisper handles the container natively — no chunking needed.
        is_wav = False
        try:
            with wave.open(io.BytesIO(audio_bytes), 'rb') as _probe:
                _probe.getnframes()
            is_wav = True
        except Exception:
            is_wav = False

        if is_wav:
            # ── WAV path: chunked processing ──────────────────────────────────────────
            fallback_starts_with = "001"
            try:
                with wave.open(io.BytesIO(audio_bytes), 'rb') as wav_in:
                    params = wav_in.getparams()
                    sr = wav_in.getframerate()
                    total_frames = wav_in.getnframes()

                frames_per_chunk = int(self.chunk_seconds * sr)
                frames_read = 0
                chunk_index = 0

                while frames_read < total_frames:
                    to_read = min(frames_per_chunk, total_frames - frames_read)

                    with wave.open(io.BytesIO(audio_bytes), 'rb') as wav_src:
                        wav_src.setpos(frames_read)
                        chunk_raw_frames = wav_src.readframes(to_read)

                    chunk_buffer = io.BytesIO()
                    with wave.open(chunk_buffer, 'wb') as wav_dst:
                        wav_dst.setparams(params)
                        wav_dst.writeframes(chunk_raw_frames)

                    chunk_payload_bytes = chunk_buffer.getvalue()
                    chunk_offset_seconds = frames_read / sr
                    chunk_filename = f"chunk_{chunk_index:03d}.wav"

                    stt_data = {
                        "model": stt_model_name, "language": self.language,
                        "stream": "false", "response_format": "verbose_json",
                        "timestamp_granularities[]": "segment"
                    }
                    stt_files = {"file": (chunk_filename, chunk_payload_bytes, "audio/wav")}
                    stt_resp = requests.post(f"{self.base_url}/audio/transcriptions", headers=headers, files=stt_files, data=stt_data, timeout=240)
                    stt_resp.raise_for_status()
                    chunk_stt_segments = stt_resp.json().get("segments", []) or []

                    diar_diarization = []
                    diar_source = "api"
                    try:
                        diar_files = {"file": (chunk_filename, chunk_payload_bytes, "audio/wav")}
                        diar_resp = requests.post(f"{self.base_url}/audio/custom-diarization", headers=headers, files=diar_files, data={"model": diar_model_name}, timeout=240)
                        if diar_resp.status_code < 400:
                            diar_diarization = diar_resp.json().get("diarization", []) or []
                        else:
                            diar_source = "fallback"
                    except Exception:
                        diar_source = "fallback"

                    if diar_source == "fallback" or not diar_diarization:
                        diar_diarization = self._alternative_diarization_from_stt(chunk_stt_segments, self.pause_threshold, start_with=fallback_starts_with)
                        diar_source = "stt_turn_fallback"
                        if diar_diarization:
                            fallback_starts_with = "002" if diar_diarization[-1].get("speaker") == "001" else "001"

                    all_aligned_segments.extend(
                        self._assign_speakers(chunk_stt_segments, diar_diarization, diar_source, chunk_offset=chunk_offset_seconds)
                    )
                    frames_read += to_read
                    chunk_index += 1

            except Exception as e:
                return f"Pipeline processing failed: {str(e)}", []

        else:
            # ── Direct path: non-WAV / video passthrough ──────────────────────────────
            # ffmpeg was unavailable in the container — send the original bytes straight
            # to WILLMA Whisper. The API handles MP4/MKV/MOV/AVI/WEBM/MP3/M4A natively.
            try:
                ext = filename.rsplit(".", 1)[-1].lower() if "." in filename else "bin"
                mime_type = self._MIME_MAP.get(ext, "application/octet-stream")

                stt_data = {
                    "model": stt_model_name, "language": self.language,
                    "stream": "false", "response_format": "verbose_json",
                    "timestamp_granularities[]": "segment"
                }
                stt_files = {"file": (filename, audio_bytes, mime_type)}
                stt_resp = requests.post(f"{self.base_url}/audio/transcriptions", headers=headers, files=stt_files, data=stt_data, timeout=600)
                stt_resp.raise_for_status()
                stt_segments = stt_resp.json().get("segments", []) or []

                diar_diarization = []
                diar_source = "api"
                try:
                    diar_files = {"file": (filename, audio_bytes, mime_type)}
                    diar_resp = requests.post(f"{self.base_url}/audio/custom-diarization", headers=headers, files=diar_files, data={"model": diar_model_name}, timeout=600)
                    if diar_resp.status_code < 400:
                        diar_diarization = diar_resp.json().get("diarization", []) or []
                    else:
                        diar_source = "fallback"
                except Exception:
                    diar_source = "fallback"

                if diar_source == "fallback" or not diar_diarization:
                    diar_diarization = self._alternative_diarization_from_stt(stt_segments, self.pause_threshold, start_with="001")
                    diar_source = "stt_turn_fallback"

                all_aligned_segments.extend(
                    self._assign_speakers(stt_segments, diar_diarization, diar_source, chunk_offset=0.0)
                )

            except Exception as e:
                return f"Pipeline processing failed: {str(e)}", []

        # ── Post-processing (shared by both paths) ────────────────────────────────────
        finalized = self._force_two_speaker_labels(all_aligned_segments)
        finalized = self._assign_unknown_by_neighbors(finalized)
        merged_conversation = self._merge_speaker_segments(finalized)

        string_script_blocks = []
        for segment in merged_conversation:
            start_min = f"{int(segment['start'] // 60)}m:{int(segment['start'] % 60):02d}s"
            speaker_tag = f"Speaker {segment['speaker']}" if segment['speaker'] != "UNKNOWN" else "Unknown Speaker"
            string_script_blocks.append(f"[{start_min}] {speaker_tag}:\n\"{segment['text'].strip()}\"")

        finalized_transcript_string = "\n\n".join(string_script_blocks)

        self._cached_result = (finalized_transcript_string, merged_conversation)
        return self._cached_result

    def get_chat_message(self) -> Message:
        text_script, _ = self.process_audio_pipeline()
        return Message(text=text_script or "Transcription returned empty.")

    def get_segments(self) -> list:
        _, raw_segments_list = self.process_audio_pipeline()
        return raw_segments_list


## Output Node: Chat Output (Built-in)

**Purpose:** Displays the transcription result in the Langflow Chat Playground.

**Action:** Langflow's built-in `ChatOutput` component (internal `lfx.*` API). Receives the `transcript_message` output from the Whisper Transcriber as a `Message` object and renders it as an AI reply in the chat panel. Handles `Message`, `Data`, `DataFrame`, and plain-string inputs. Applies a configurable `data_template` for `Data` inputs, stores messages to session history when enabled, and attaches accumulated upstream LLM token-usage metadata to the stored message after streaming.

In [ ]:
from collections.abc import Generator
from typing import Any

import orjson
from fastapi.encoders import jsonable_encoder

from lfx.base.io.chat import ChatComponent
from lfx.helpers.data import safe_convert
from lfx.inputs.inputs import BoolInput, DropdownInput, HandleInput, MessageTextInput
from lfx.schema.data import Data
from lfx.schema.dataframe import DataFrame
from lfx.schema.message import Message
from lfx.schema.properties import Source
from lfx.template.field.base import Output
from lfx.utils.constants import (
    MESSAGE_SENDER_AI,
    MESSAGE_SENDER_NAME_AI,
    MESSAGE_SENDER_USER,
)


class ChatOutput(ChatComponent):
    display_name = "Chat Output"
    description = "Display a chat message in the Playground."
    documentation: str = "https://docs.langflow.org/chat-input-and-output"
    icon = "MessagesSquare"
    name = "ChatOutput"
    minimized = True

    inputs = [
        HandleInput(
            name="input_value",
            display_name="Inputs",
            info="Message to be passed as output.",
            input_types=["Data", "JSON", "DataFrame", "Table", "Message"],
            required=True,
        ),
        BoolInput(
            name="should_store_message",
            display_name="Store Messages",
            info="Store the message in the history.",
            value=True,
            advanced=True,
        ),
        DropdownInput(
            name="sender",
            display_name="Sender Type",
            options=[MESSAGE_SENDER_AI, MESSAGE_SENDER_USER],
            value=MESSAGE_SENDER_AI,
            advanced=True,
            info="Type of sender.",
        ),
        MessageTextInput(
            name="sender_name",
            display_name="Sender Name",
            info="Name of the sender.",
            value=MESSAGE_SENDER_NAME_AI,
            advanced=True,
        ),
        MessageTextInput(
            name="session_id",
            display_name="Session ID",
            info="The session ID of the chat. If empty, the current session ID parameter will be used.",
            advanced=True,
        ),
        MessageTextInput(
            name="context_id",
            display_name="Context ID",
            info="The context ID of the chat. Adds an extra layer to the local memory.",
            value="",
            advanced=True,
        ),
        MessageTextInput(
            name="data_template",
            display_name="Data Template",
            value="{text}",
            advanced=True,
            info="Template to convert Data to Text. If left empty, it will be dynamically set to the Data's text key.",
        ),
        BoolInput(
            name="clean_data",
            display_name="Basic Clean Data",
            value=True,
            advanced=True,
            info="Whether to clean data before converting to string.",
        ),
    ]
    outputs = [
        Output(
            display_name="Output Message",
            name="message",
            method="message_response",
        ),
    ]

    def _build_source(self, id_: str | None, display_name: str | None, source: str | None) -> Source:
        source_dict = {}
        if id_:
            source_dict["id"] = id_
        if display_name:
            source_dict["display_name"] = display_name
        if source:
            # Handle case where source is a ChatOpenAI object
            if hasattr(source, "model_name"):
                source_dict["source"] = source.model_name
            elif hasattr(source, "model"):
                source_dict["source"] = str(source.model)
            else:
                source_dict["source"] = str(source)
        return Source(**source_dict)

    async def message_response(self) -> Message:
        # First convert the input to string if needed
        text = self.convert_to_string()

        # Get source properties
        source, _, display_name, source_id = self.get_properties_from_source_component()

        # Create or use existing Message object
        if isinstance(self.input_value, Message) and not self.is_connected_to_chat_input():
            message = self.input_value
            # Update message properties
            message.text = text
            # Preserve existing session_id from the incoming message if it exists
            existing_session_id = message.session_id
        else:
            message = Message(text=text)
            existing_session_id = None

        # Set message properties
        message.sender = self.sender
        message.sender_name = self.sender_name
        # Preserve session_id from incoming message, or use component/graph session_id
        message.session_id = (
            self.session_id or existing_session_id or (self.graph.session_id if hasattr(self, "graph") else None) or ""
        )
        message.context_id = self.context_id
        message.flow_id = self.graph.flow_id if hasattr(self, "graph") else None
        message.properties.source = self._build_source(source_id, display_name, source)

        # Store message if needed
        if message.session_id and self.should_store_message:
            stored_message = await self.send_message(message)
            self.message.value = stored_message
            message = stored_message

        # Set accumulated token usage from all upstream LLM vertices.
        # This must happen AFTER send_message() because streaming captures
        # usage from chunks and would overwrite accumulated totals.
        if hasattr(self, "_vertex") and self._vertex is not None:
            accumulated_usage = self._vertex._accumulate_upstream_token_usage()  # noqa: SLF001
            if accumulated_usage:
                message.properties.usage = accumulated_usage
                if self.should_store_message and message.get_id():
                    message = await self._update_stored_message(message)
                    await self._send_message_event(message, id_=message.get_id())

        self.status = message
        return message

    def _serialize_data(self, data: Data) -> str:
        """Serialize Data object to JSON string."""
        # Convert data.data to JSON-serializable format
        serializable_data = jsonable_encoder(data.data)
        # Serialize with orjson, enabling pretty printing with indentation
        json_bytes = orjson.dumps(serializable_data, option=orjson.OPT_INDENT_2)
        # Convert bytes to string and wrap in Markdown code blocks
        return "```json\n" + json_bytes.decode("utf-8") + "\n```"

    def _validate_input(self) -> None:
        """Validate the input data and raise ValueError if invalid."""
        if self.input_value is None:
            msg = "Input data cannot be None"
            raise ValueError(msg)
        if isinstance(self.input_value, list) and not all(
            isinstance(item, Message | Data | DataFrame | str) for item in self.input_value
        ):
            invalid_types = [
                type(item).__name__
                for item in self.input_value
                if not isinstance(item, Message | Data | DataFrame | str)
            ]
            msg = f"Expected Data or DataFrame or Message or str, got {invalid_types}"
            raise TypeError(msg)
        if not isinstance(
            self.input_value,
            Message | Data | DataFrame | str | list | Generator | type(None),
        ):
            type_name = type(self.input_value).__name__
            msg = f"Expected Data or DataFrame or Message or str, Generator or None, got {type_name}"
            raise TypeError(msg)

    def convert_to_string(self) -> str | Generator[Any, None, None]:
        """Convert input data to string with proper error handling."""
        self._validate_input()
        if isinstance(self.input_value, list):
            clean_data: bool = getattr(self, "clean_data", False)
            return "\n".join([safe_convert(item, clean_data=clean_data) for item in self.input_value])
        if isinstance(self.input_value, Generator):
            return self.input_value
        return safe_convert(self.input_value)


---

## Flow Summary — WILLMA Whisper V07

### What this flow does

Accepts a public **audio or video file URL** in the Langflow Chat Playground, downloads the file into RAM, optionally extracts the audio track from video containers via `ffmpeg`, cleans the audio with a **pure-Python DSP chain**, and sends it to the SURF WILLMA Whisper API for **transcription + two-speaker diarization**. Returns a timestamped `[Xm:YYs] Speaker 001/002:` dialogue script to the Chat Playground.

---

### Pipeline

| Step | Component | Input | Output |
|------|-----------|-------|--------|
| 1 | **Chat Input** (built-in) | User-typed audio/video URL | `Message` wrapping the URL text |
| 2 | **WILLMA Audio Downloader** | `Message` | `Data` with `bytes`, `filename`, `size_bytes` |
| 3 | **Audio Preprocessor** | `Data` (raw audio or video bytes) | `Data` (cleaned 16 kHz mono PCM WAV) |
| 4 | **WILLMA Whisper Diarized Transcriber** | `Data` (cleaned audio) | `Message` (dialogue script) + `list` (segments) |
| 5 | **Chat Output** (built-in) | `Message` | Dialogue script displayed in Playground |

---

### Key design decisions

| Decision | Rationale |
|----------|-----------|
| ffmpeg via /tmp temp file | Writes video bytes to a named temp file for ffmpeg — gives full random-access seeking required for moov-at-end MP4s (Zoom, phone captures). Avoids cache:pipe:0 failures inside Docker. |
| ffmpeg outputs 16 kHz mono 16-bit directly | The subsequent resampling DSP step is a no-op for video-extracted audio — no double-processing penalty |
| Resilient passthrough on ffmpeg failure | If `ffmpeg` is not installed or returns non-zero, raw bytes are forwarded to Whisper unchanged rather than crashing the flow |
| Extension-based video detection | Avoids reading magic bytes or probing the container; keeps the detection path in stdlib |
| Stdlib-only DSP preprocessor | No scipy / numpy dependency — runs in any Python environment without native extensions |
| Linear interpolation resampling | Pure Python; no C extension or CFFI required |
| Power-law DRC (`\|s\|^0.88`) | Simple soft-knee compression with no look-ahead, attack/release, or RMS window needed |
| Passthrough for non-WAV / non-16bit | Avoids decoding formats the flow has no codec for; Whisper handles them natively |
| 30 s chunk windows | Keeps individual API payloads well under the WILLMA size limit; allows interleaved STT + diarization per chunk |
| Overlap-matrix speaker alignment | Assigns the speaker with the highest time-overlap to each STT segment — tolerates misaligned boundaries |
| STT-turn fallback diarization | Pause-gap turn detection keeps the flow working when the pyannote API is unavailable |
| `_cached_result` in transcriber | Both output methods share a single pipeline run — no double API billing |
| Built-in `ChatInput` / `ChatOutput` | Uses Langflow's internal registry for playground routing without reimplementing the public API |

---

### Configuration

Before running, set the **WILLMA API Key** (`api_key`) field in the Diarized Transcriber component.
All other parameters have sensible defaults:

| Parameter | Default | Where |
|-----------|---------|-------|
| `target_sample_rate` | 16 000 Hz | Preprocessor |
| `highpass_cutoff_hz` | 90 Hz | Preprocessor |
| `enable_drc` | `True` | Preprocessor |
| `peak_normalize` | 0.98 | Preprocessor |
| `base_url` | `https://willma.surf.nl/api/v0` | Transcriber |
| `language` | `nl` | Transcriber |
| `chunk_seconds` | 30 s | Transcriber |
| `min_overlap_seconds` | 0.15 s | Transcriber |
| `pause_threshold` | 1.2 s | Transcriber |

---

### Supported input formats

| Format | Preprocessor action | Passed to Whisper |
|--------|-------------------|-------------------|
| MP4, MKV, MOV, AVI, WEBM, M4V, MPG | **ffmpeg audio extraction** → 16 kHz mono WAV → DSP chain | ✅ |
| 16-bit PCM WAV | Full DSP chain (resample, filter, DRC, normalize) | ✅ |
| WAV with other bit-depth | Passthrough (no DSP) | ✅ |
| MP3, M4A, FLAC, OGG | Passthrough (no DSP) | ✅ (Whisper handles natively) |

> **Docker requirement:** `ffmpeg` must be installed in the Langflow container.
> Add to your `Dockerfile` (before `USER 1000`):
> ```dockerfile
> RUN apt-get update && apt-get install -y ffmpeg && rm -rf /var/lib/apt/lists/*
> ```
> After editing the Dockerfile, rebuild: `docker compose up -d --build`

---

### Langflow flow file

A working V06 base flow (without the V07 video-extraction update) is available at:

```
D:\OneDrive - Hogeschool Rotterdam\SURF_PILOT\AI_HUB_PILOT\PYTHON\URL PREPRO TRANSCR  + TIME + DIARIZATION.json
```

To create a V07 flow file:
1. Import the V06 JSON via **Langflow → My Flows → Import**.
2. Open the flow, select the **2. Audio Preprocessor** component, click **Edit Code**, and replace its code with the updated `AudioPreprocessorComponent` from cell 8 of this notebook.
3. Click **Save**, then export via **⋮ → Export** to save the updated flow as a new JSON file.